# ML-08 — Capstone Modeling Lane

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanraza04/flyrank_intern_content/blob/capstone-refresh/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook compares the transparent baseline with two classifiers on the same time-aware cohorts. Run it in the same Colab runtime as ML-07 so it can use the cached page-level feature frame.

## 1. Method choice and why

This is a scoring and ranking task with an observed, later-window proxy label. I start with regularized logistic regression because it is readable, then compare a shallow histogram gradient-boosting model that can learn modest non-linear patterns. The model is useful only if it beats the frozen baseline on the same later cohort.

In [ ]:
%pip -q install pandas pyarrow scikit-learn matplotlib

from pathlib import Path

if not Path('work/scripts/capstone_utils.py').exists():
    !git clone -q --branch capstone-refresh https://github.com/hassanraza04/flyrank_intern_content.git
    %cd flyrank_intern_content


## 2. Split design

The split is time-aware. Training uses outcomes that end no later than March 22, 2026. The May development cohort is used to select a method. Only after selection do I evaluate the sealed June outcome cohort. This mirrors the decision sequence and prevents the model from training on future outcomes.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from work.scripts.capstone_utils import (
    FEATURE_COLUMNS,
    add_baseline_score,
    evaluate_ranking,
    validate_feature_columns,
)

CACHE_PATH = Path('work/outputs/capstone_features.parquet')
if not CACHE_PATH.exists():
    raise RuntimeError('Run w04_baseline_score.ipynb in this Colab runtime first.')

features = pd.read_parquet(CACHE_PATH)
validate_feature_columns(FEATURE_COLUMNS)

train = features[features['cohort_id'].isin(['2025-12', '2026-01', '2026-02', '2026-03', '2026-04'])].copy()
validation = features[features['cohort_id'] == '2026-05'].copy()
sealed = features[features['cohort_id'] == '2026-06-sealed'].copy()

print({'train_rows': len(train), 'validation_rows': len(validation), 'sealed_rows': len(sealed)})

## 3. Train and compare versus my baseline

Every method uses the same validation rows and reports Precision@100, ROC-AUC, and the proxy base rate. The baseline is already frozen in ML-07.

In [ ]:
models = {
    'logistic_regression': make_pipeline(
        SimpleImputer(strategy='median'),
        LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    ),
    'hist_gradient_boosting': make_pipeline(
        SimpleImputer(strategy='median'),
        HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08, max_iter=150, random_state=42),
    ),
}

validation_results = []
for name, model in models.items():
    model.fit(train[FEATURE_COLUMNS], train['is_declining_proxy'])
    scores = model.predict_proba(validation[FEATURE_COLUMNS])[:, 1]
    validation_results.append({'method': name, **evaluate_ranking(validation['is_declining_proxy'], scores, k=100)})

validation_baseline = add_baseline_score(validation)
validation_results.append({
    'method': 'momentum_baseline',
    **evaluate_ranking(validation_baseline['is_declining_proxy'], validation_baseline['baseline_score'], k=100),
})

comparison = pd.DataFrame(validation_results).sort_values('precision_at_k', ascending=False).reset_index(drop=True)
comparison

In [ ]:
selected_name = comparison.loc[0, 'method']
all_development = pd.concat([train, validation], ignore_index=True)

if selected_name == 'momentum_baseline':
    selected_model = None
else:
    selected_model = models[selected_name].fit(all_development[FEATURE_COLUMNS], all_development['is_declining_proxy'])

print(f'Frozen choice before sealed evaluation: {selected_name}')

sealed_baseline = add_baseline_score(sealed)
baseline_metrics = evaluate_ranking(sealed['is_declining_proxy'], sealed_baseline['baseline_score'], k=100)
if selected_model is None:
    sealed_scores = sealed_baseline['baseline_score']
else:
    sealed_scores = selected_model.predict_proba(sealed[FEATURE_COLUMNS])[:, 1]
selected_metrics = evaluate_ranking(sealed['is_declining_proxy'], sealed_scores, k=100)

metrics_receipt = {
    'cohort': '2026-06-sealed',
    'target': 'next_28d_impressions < 0.80 * current_28d_impressions',
    'metric_k': 100,
    'feature_columns': FEATURE_COLUMNS,
    'selected_method': selected_name,
    'baseline': baseline_metrics,
    'selected_method_metrics': selected_metrics,
    'random_seed': 42,
}
print(json.dumps(metrics_receipt, indent=2))

## 4. Errors and interpretation

The top-ranked pages should be treated as review candidates, not proof that a refresh will improve performance. Important error sources include seasonality, changes in demand, indexing changes, and sparse recent history. A small lift over the baseline is still useful only if editors find the reason codes plausible during review.

In [ ]:
selected_metrics['base_rate'], selected_metrics['precision_at_k'], selected_metrics['roc_auc']


## Self-check

- [x] The split is chronological and the final June outcome cohort is sealed until method selection.
- [x] Baseline and model use the same candidates, label, and Precision@100 metric.
- [x] Every metric has its proxy base rate beside it.
- [x] IDs, outcome fields, and future values are not model inputs.
- [ ] I have run the notebook and saved the executed result to GitHub.
